<a href="https://colab.research.google.com/github/malmakt000/Data-Based/blob/main/models/Housing_Risk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay



In [ ]:
API_KEY="d48d60e30134e472760fbe35a9f408a3f03d6fdf"
url=f"https://api.census.gov/data/2024/acs/acs1/pums?get=HINCP,PUMA,GRPIP,NP&for=state:36&key={API_KEY}"

# fecth data
response=requests.get(url)
print(response.status_code)

200


In [ ]:
# Create dataframe
data=response.json()
df=pd.DataFrame(data[1:],columns=data[0])
df.head()
df.min()
df.max()

#HINCP-Household Income: The total self-reported annual income for every person living in the household
#PUMA-Public Use Microdata Area: A 5-digit geographic code, specific to Community Districts
#GRPIP - Gross Rent as a Percentage of Household Income: Calculates how much of the monthly income goes toward rent and utilities
#State-New York State Code: The standard FIPS(Federal Information Processing Series) code for NY

#print(df.columns.tolist())

# Filter out nonsensical records
#Convert everything to numbers

#print(df.isnull().sum())

df['HINCP'] = pd.to_numeric(df['HINCP'])
df['GRPIP'] = pd.to_numeric(df['GRPIP'])
df['PUMA'] = pd.to_numeric(df['PUMA'])
df['state']=pd.to_numeric(df['state'])
df['NP']=pd.to_numeric(df['NP'])

# 2. Apply Filters to get rid of incomplete dat
df_nyc = df[
    (df["HINCP"] >= 10000) & # Focus on working households (min wage floor and max 300,000)
    (df["HINCP"] <= 300000) &
    (df["GRPIP"] > 0) &         # Must be a renter
    (df["GRPIP"] < 99) &        # Remove the '99' Census cap/outliers
    (df["PUMA"] >= 3701) &      # Filter for BX, Manhattan, BK, Queens, SI
    (df["PUMA"] <= 4114)
]

df_nyc=df_nyc.copy()



# Calculations for monthly rent and monthly income
df_nyc["monthly_income"]=(df_nyc["HINCP"])/12
df_nyc["monthly_rent"]=(df_nyc["monthly_income"]*(df_nyc["GRPIP"]))/100

df_nyc["rent"]=df_nyc["monthly_rent"]
df_nyc["rent"]=df_nyc["rent"].round(1)

#DEBUG STUFF
#print(df_nyc[["HINCP", "GRPIP", "rent"]].head())
# #print(f"Cleaned rows: {len(df_nyc)}")
# #print(df_nyc.head())
# df_nyc.hist(figsize=(10,8))
# #df_nyc.describe()
# print(df_nyc.max())
# print()
# print(df_nyc.min())
# print()

#Define the base cost of living for 1 person and the 'additional person' cost
# Based on the Numbeo $1,675 for 1 person vs $6,215 for 4 ppl difference
base_living_cost = 1675.60
add_person_cost = 1513.00

# Calculate  living costs based on household size (NP)
df_nyc['living_costs'] = base_living_cost + ((df_nyc['NP'] - 1) * add_person_cost)

# Calculate the total Financial Burden
df_nyc['total_burden'] = df_nyc['rent'] + df_nyc['living_costs']

# Caluctae Disposable Income
df_nyc['disposable_income'] = df_nyc['monthly_income'] - df_nyc['total_burden']

print(df_nyc[['NP', 'monthly_income', 'total_burden', 'disposable_income']].head())

# Standard Rent Burden (already in your data as GRPIP)
df_nyc['rent_burden'] = df_nyc['GRPIP'] / 100

#Total Cost Burden = cost of living/montly income
df_nyc['cost_burden'] = df_nyc['total_burden'] / df_nyc['monthly_income']

# Classigy Risk Score
# 7 and above is high risk, if dispaobke income is negative automatically high risk
# 5 - 7 medium risk
# 5<=low risk
def classify_risk(row):
    if row["cost_burden"] > 0.7 or row["disposable_income"] < 0:
        return 2  # High risk
    elif row["cost_burden"] > 0.5:
        return 1  # Medium risk
    else:
        return 0  # Low risk

df_nyc["risk_label"] = df_nyc.apply(classify_risk, axis=1)


# USE IMPORTANT FEATURES we are not using cost burden and discposible income bc it is used to calcuate the risk score

fair_features = [
    "monthly_income",
    "rent",
    "rent_burden",
    "NP",
    "PUMA"
]

X = df_nyc[fair_features]
y = df_nyc["risk_label"] # Target class

# PUMA dada is catergorical, covert to raw numbers use one hot ecnoding

X = pd.get_dummies(X, columns=["PUMA"], drop_first=True)


# Train model using Random Forest
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

df_results = X_test.copy()
df_results["true"] = y_test
df_results["pred"] = y_pred

# DEUBUG
print(df_results.head(10))

# Check theAccuracy
y_pred_f = fair_model.predict(X_test_f)
print("--- REAL WORLD ACCURACY ---")
print(classification_report(y_test_f, y_pred_f))





# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Low', 'Medium', 'High'])
disp.plot(cmap='Blues', values_format='d')
plt.title('Confusion Matrix: Housing Risk Prediction')
plt.grid(False) # Clean up the grid lines
plt.savefig('confusion_matrix.png')
print("Confusion Matrix saved as 'confusion_matrix.png'")

# Feature Importance Graph
importances = model.feature_importances_
feature_names = X.columns
feature_importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})

# Sort by importance and take the top 10
feature_importance_df = feature_importance_df.sort_values(by='Importance', ascending=False).head(5)

plt.figure(figsize=(10, 6))
sns.barplot(x='Importance', y='Feature', data=feature_importance_df, palette='viridis')
plt.title('Top 10 Most Important Features for Predicting Housing Risk')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.savefig('feature_importance.png')
print("Feature Importance graph saved as 'feature_importance.png'")



       NP  monthly_income  total_burden  disposable_income
16236   2     1250.000000        4163.6       -2913.600000
16300   1     4333.333333        2585.6        1747.733333
16301   2     7925.000000        4139.6        3785.400000
16316   2    15000.000000        6788.6        8211.400000
16341   3     1050.000000        4806.6       -3756.600000
Accuracy: 0.9414893617021277
              precision    recall  f1-score   support

           0       0.95      0.95      0.95       126
           1       0.86      0.88      0.87        85
           2       0.98      0.96      0.97       165

    accuracy                           0.94       376
   macro avg       0.93      0.93      0.93       376
weighted avg       0.94      0.94      0.94       376

        monthly_income    rent  rent_burden  NP  PUMA_4104  PUMA_4107  \
66918     13333.333333  6666.7         0.50   1      False      False   
83028      3266.666667  2744.0         0.84   3      False      False   
77260      5633.3

NameError: name 'fair_model' is not defined